# ローマ字をかなに変換する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/31_japanese_input/romaji_to_kana.ipynb)

NLP クックブック 31-01 ／ 日本語入力（かな漢字変換）

キーボードから打たれたローマ字の列を、ひらがなに変換したい。促音（っ）や撥音（ん）の規則も正しく扱いたい。

In [1]:
!pip install -q jaconv

In [2]:
# 変換表: 長い綴りを優先して照合する (最長一致)
TABLE = {
    "a": "あ", "i": "い", "u": "う", "e": "え", "o": "お",
    "ka": "か", "ki": "き", "ku": "く", "ke": "け", "ko": "こ", "sa": "さ", "si": "し", "shi": "し", "su": "す", "se": "せ", "so": "そ",
    "ta": "た", "ti": "ち", "chi": "ち", "tu": "つ", "tsu": "つ", "te": "て", "to": "と", "na": "な", "ni": "に", "nu": "ぬ", "ne": "ね", "no": "の",
    "ha": "は", "hi": "ひ", "hu": "ふ", "fu": "ふ", "he": "へ", "ho": "ほ", "ma": "ま", "mi": "み", "mu": "む", "me": "め", "mo": "も",
    "ya": "や", "yu": "ゆ", "yo": "よ", "ra": "ら", "ri": "り", "ru": "る", "re": "れ", "ro": "ろ", "wa": "わ", "wo": "を", "nn": "ん",
    "ga": "が", "gi": "ぎ", "gu": "ぐ", "ge": "げ", "go": "ご", "za": "ざ", "zi": "じ", "ji": "じ", "zu": "ず", "ze": "ぜ", "zo": "ぞ",
    "da": "だ", "di": "ぢ", "du": "づ", "de": "で", "do": "ど", "ba": "ば", "bi": "び", "bu": "ぶ", "be": "べ", "bo": "ぼ",
    "pa": "ぱ", "pi": "ぴ", "pu": "ぷ", "pe": "ぺ", "po": "ぽ", "-": "ー", ",": "、", ".": "。",
    "la": "ぁ", "li": "ぃ", "lu": "ぅ", "le": "ぇ", "lo": "ぉ", "xtu": "っ", "ltu": "っ",
}
for c, kana in [("k", "き"), ("s", "し"), ("t", "ち"), ("n", "に"), ("h", "ひ"), ("m", "み"), ("r", "り"), ("g", "ぎ"), ("z", "じ"), ("b", "び"), ("p", "ぴ"), ("d", "ぢ")]:
    for v, small in [("a", "ゃ"), ("u", "ゅ"), ("o", "ょ")]:
        TABLE[c + "y" + v] = kana + small                      # kya → きゃ など
for a, kana in [("sh", "し"), ("ch", "ち"), ("j", "じ")]:
    for v, small in [("a", "ゃ"), ("u", "ゅ"), ("o", "ょ")]:
        TABLE[a + v] = kana + small                            # sha → しゃ、ja → じゃ など
MAX_LEN = max(map(len, TABLE))

def romaji_to_kana(text):
    text, out, i = text.lower(), [], 0
    while i < len(text):
        # 促音: 同じ子音が続く (kk, tt, ss ...)。ただし nn は「ん」。ヘボン式の tch (matcha) も促音
        if i + 1 < len(text) and (text[i] == text[i + 1] and text[i] in "kstpgzdbhfmrwcj" or text[i:i + 3] == "tch"):
            out.append("っ"); i += 1; continue
        for n in range(MAX_LEN, 0, -1):                        # 最長一致
            if text[i:i + n] in TABLE:
                out.append(TABLE[text[i:i + n]]); i += n; break
        else:
            if text[i] == "n" and (i + 1 == len(text) or text[i + 1] not in "aiueoy"):
                out.append("ん")                               # 母音や y が続かない n は「ん」
            else:
                out.append(text[i])                            # 変換できない文字はそのまま (入力途中の子音など)
            i += 1
    return "".join(out)

for s in ["kyouhaiitenkidesu", "gakkouniitta", "shinbunwoyomu", "konnnichiha", "kinen", "kinnen", "matcha", "chottomatte", "ko-hi-", "nihongonyuuryoku", "sizengengosyori", "k"]:
    print(f"{s:20} → {romaji_to_kana(s)}")

kyouhaiitenkidesu    → きょうはいいてんきです
gakkouniitta         → がっこうにいった
shinbunwoyomu        → しんぶんをよむ
konnnichiha          → こんにちは
kinen                → きねん
kinnen               → きんえん
matcha               → まっちゃ
chottomatte          → ちょっとまって
ko-hi-               → こーひー
nihongonyuuryoku     → にほんごにゅうりょく
sizengengosyori      → しぜんげんごしょり
k                    → k


In [3]:
# ライブラリを使う場合 / 逆方向 (かな → ローマ字)
import jaconv

for s in ["kyouhaiitenkidesu", "gakkouniitta", "kinen", "kinnen", "chottomatte"]:
    print(f"{s:20} → {jaconv.alphabet2kana(s)}")
print(jaconv.kana2alphabet("にほんごにゅうりょく"), "/", jaconv.kata2alphabet("カナカンジヘンカン"))

kyouhaiitenkidesu    → きょうはいいてんきです
gakkouniitta         → がっこうにいった
kinen                → きねん
kinnen               → きんえん
chottomatte          → ちょっとまって
nihongonyuuryoku / kanakanjihenkan


- 1 つのかなに複数の綴り（si / shi、tu / tsu、zi / ji）があり、利用者によって違う。実際の入力システムは、ヘボン式・訓令式・独自の綴りをすべて受け付ける表を持つ。
- 「ん」の扱いが難しい。`kinen` は「きねん」、`kinnen` は「きんえん」、`konnichiha` と `konnnichiha` は利用者の習慣で分かれる。次の文字を見ないと決められない。
- 入力途中の未確定の子音（最後の `k`）は、変換せずに残しておく。1 文字打つたびに先頭から変換し直せば、表示は常に正しくなる。
- 表を差し替えれば、かな入力や AZIK などの拡張ローマ字入力にも同じ仕組みで対応できる。
- 漢字を含む文をローマ字にするには、文字列処理と正規化の章の「漢字かな交じり文をローマ字・ひらがなにする」を参照。